# VisionBridge: improved BLIP fine-tuning on a Colab T4

This is a new workflow. Vidish's `mai_model_train.ipynb` and the existing
`models/blip_vizwiz` checkpoint remain unchanged. Fine-tuning starts from that
verified checkpoint. New models are saved in a separate Google Drive run folder.

Select **Runtime → Change runtime type → T4 GPU**. Run one section at a time.
Data preparation and training have explicit switches, initially disabled.
The notebook uses `visionbridge_training.py` and `evaluate_vqa.py`, so there is
one implementation to maintain. Upload your local changed files: they are not
yet committed or available on GitHub.


## 1. Install loading dependencies

Keep Colab's CUDA PyTorch. If you already imported Transformers or Pillow,
restart the session after installation, then continue below.


In [ ]:
import subprocess, sys, json
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'transformers==4.57.6', 'tokenizers==0.22.2',
    'huggingface-hub>=0.34,<1.0', 'safetensors>=0.4.3,<0.9',
    'Pillow>=11.1,<12', 'matplotlib>=3.8,<4'])

## 2. Retrieve the original checkpoint and upload the improved code

GitHub supplies the unchanged checkpoint through Git LFS (~1.45 GB).
The local development branch is not pushed; cloning only supplies the original
checkpoint. Upload **four local files** from your project folder:
`colab_train_t4.py`, `visionbridge_training.py`, `evaluate_vqa.py`,
and `requirements-training.txt`. Code goes into a new temporary folder, so a
setup rerun cannot overwrite a previous upload. This notebook never runs the
original training notebook. If you change code after importing it, restart the
session before continuing; do not change code mid-experiment.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, uuid
from google.colab import files

REPO = Path('/content/visionbridge_repo')
CODE = Path('/content/visionbridge_code') / uuid.uuid4().hex[:8]
if shutil.which('git-lfs') is None:
    subprocess.check_call(['apt-get', 'update', '-qq'])
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'git-lfs'])
if not REPO.exists():
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
    subprocess.check_call(['git', 'clone', 'https://github.com/vedants27/mai-project.git', str(REPO)], env=env)
else:
    assert (REPO / '.git').is_dir(), 'Choose another directory; do not overwrite existing files.'
subprocess.check_call(['git', 'lfs', 'install', '--local'], cwd=REPO)
subprocess.check_call(['git', 'lfs', 'pull', 'origin', '--include=models/blip_vizwiz/model.safetensors'], cwd=REPO)
CODE.mkdir(parents=True, exist_ok=False)
os.chdir(CODE)
uploaded = files.upload()
SOURCE_FILES = ('colab_train_t4.py', 'visionbridge_training.py', 'evaluate_vqa.py', 'requirements-training.txt')
assert set(SOURCE_FILES) <= set(uploaded), 'Upload all four named files.'
sys.path.insert(0, str(CODE))
for name in ('colab_train_t4', 'visionbridge_training', 'evaluate_vqa'):
    sys.modules.pop(name, None)

## 3. Choose a persistent run

Defaults: 5,000 training examples, 1,000 development examples, batch 4 with
four accumulation steps (effective batch 16), learning rate 1e-5, at most five
epochs, and two development checks without a meaningful improvement before
stopping. Every epoch measures VQA accuracy and validation loss. The highest
development score selects the best model, including the starting checkpoint
if training never improves it. Use a new RUN_NAME for a different experiment.

Latest resumable snapshots are saved every 500 successful updates and each
epoch. They include trainable weights, optimizer, scheduler, FP16 scaler,
random state and the exact next batch. Snapshots can take several hundred MB
each; reserve at least 10 GB free in Drive. Old snapshots are not automatically deleted.

For a disconnected runtime, set RESUME=True and use the same RUN_NAME. Run
setup and preparation again to restore local images. The time budget is 120
minutes per training invocation, excluding setup, preparation and initial
baseline evaluation. Checkpoint saving/evaluation can finish after the deadline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, transformers, matplotlib, platform
from visionbridge_training import (TrainConfig, create_run, read_json, sha256,
    prepare_official_data, load_model, configure_trainable, make_batch,
    train_run, final_compare, write_json, check_run)
from evaluate_vqa import OfficialScorer

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
assert Path('/content/drive/MyDrive').is_dir(), 'Drive must be mounted; never save runs to an unmounted local path.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)
RUN_NAME = 'vizwiz-5000-lr1e5-run02'  # New name for every new experiment.
RESUME = False  # True only for returning to this same existing run.
RUN_DIR = Path('/content/drive/MyDrive/VisionBridge/runs') / RUN_NAME
DATA_ROOT = Path('/content/visionbridge_data')
INIT = REPO / 'models/blip_vizwiz'
expected = 'ca9f681f602f55d9d20868559e11eae4cf2be7086efd26d002e05e984f2f8099'
assert sha256(INIT / 'model.safetensors') == expected, 'Checkpoint integrity mismatch.'
code_hashes = {name: sha256(CODE / name) for name in SOURCE_FILES}
if RESUME:
    config = TrainConfig(**read_json(RUN_DIR / 'config.json'))
    check_run(config, RUN_DIR)
    assert read_json(RUN_DIR / 'code_hashes.json') == code_hashes, 'Resume with the exact saved code in RUN_DIR/code_source.'
else:
    config = TrainConfig(init_checkpoint=str(INIT))
    create_run(config, RUN_DIR)  # Refuses to reuse an existing run.
    write_json(RUN_DIR / 'code_hashes.json', code_hashes)
    source_dir = RUN_DIR / 'code_source'
    source_dir.mkdir(exist_ok=False)
    for name in SOURCE_FILES:
        with (source_dir / name).open('xb') as target:
            target.write((CODE / name).read_bytes())
    write_json(RUN_DIR / 'environment.json', {
        'python': platform.python_version(), 'torch': torch.__version__,
        'transformers': transformers.__version__, 'matplotlib': matplotlib.__version__,
        'gpu': torch.cuda.get_device_name(0)})
print(config)
print('Persistent output:', RUN_DIR)
print('Before proceeding, confirm at least 10 GB free in your Google Drive storage page.')

## 4. Prepare reproducible official data

This explicitly downloads the official train and validation image archives,
(~11.30 GB train + 3.49 GB validation) even though only 5,000 training images are extracted.
Archives and active images stay on the Colab disk; checkpoints stay on Drive.
Check free disk space before enabling. Downloads reuse complete files and keep
interrupted downloads under unique names. Existing images are checked and
never overwritten. There is no additional JPEG resize/recompression.

Sampling uses complete annotation lists and preserves answerability/answer-type
proportions. All ten answers remain in the manifest. Of 4,319 validation
examples, 1,000 select checkpoints; 3,319 are reserved for final comparison.
The existing model may have been inspected on some validation examples already,
so this is a holdout for the new experiment, not proof of a never-seen benchmark.
The official API archive (~186 MB, including optional caption assets) supplies
normalization tables. No downloaded API Python code is executed.

Reserve about 30 GB of local disk for archives, extracted images and models.
Partial downloads/extractions are retained under unique names; incomplete files
are never published as completed images. Checkpoints, manifests, saved code,
metrics and graphs all live in the mounted Drive run folder.

Reference policy: preserve all ten original annotation slots and valid answer
frequencies. Blank/non-string/malformed individual responses cannot become
targets or scoring matches. Examples with no usable references are excluded
before sampling in every split, and reported with their original answerability
in split_manifest.json. For the existing run01 annotation error, use a fresh
run02 with RESUME=False; keep run01 intact because its saved code hashes identify
the previous loader. See DATASET_FIX_GUIDE.md for updating an existing session.


In [ ]:
local_free = shutil.disk_usage('/content').free / 1e9
print('Local free GB:', round(local_free, 1))
PREPARE_DATA = False  # Set True when ready to download the dataset in Colab.
if PREPARE_DATA:
    assert local_free >= 30, 'Reserve about 30 GB local disk for archives, images and models.'
    manifest = prepare_official_data(config, RUN_DIR, DATA_ROOT)
    print({split: len(manifest[split]) for split in ('train', 'dev', 'holdout')})
    print('Reference policy:', manifest['reference_policy'])
    print('Annotation audit:', json.dumps(manifest['annotation_audit'], indent=2))
else:
    print('Preparation disabled. Enable before the following checks.')

## 5. Lightweight real-model check (no optimizer update)

Confirm local checkpoint loading, answer padding, a finite evaluation loss,
and generated answers before starting training. The decoder input IDs contain
real tokens; only labels contain the -100 loss mask. Do not shift labels again:
BLIP's loss already aligns next-token predictions.


In [ ]:
assert (RUN_DIR / 'split_manifest.json').exists(), 'Prepare data first.'
scorer = OfficialScorer.from_file(RUN_DIR / 'official_normalization.json')
manifest = read_json(RUN_DIR / 'split_manifest.json')
processor, check_model = load_model(config.init_checkpoint, 'cuda')
configure_trainable(check_model, config.decoder_layers)
check_model.eval()
batch = make_batch(manifest['dev'][:2], DATA_ROOT / 'images', processor,
                   check_model, config, 'cuda', targets=True)
assert (batch['decoder_input_ids'] >= 0).all()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    loss = check_model(**batch).loss
    inputs = {k: v for k, v in batch.items() if k not in
              {'labels', 'decoder_input_ids', 'decoder_attention_mask'}}
    generated = check_model.generate(**inputs, max_new_tokens=config.max_new_tokens,
                                     num_beams=1, do_sample=False)
assert torch.isfinite(loss)
print('Finite evaluation loss:', loss.item())
print('Sample answers:', processor.batch_decode(generated, skip_special_tokens=True))
del check_model, processor, batch, inputs, generated
import gc
gc.collect()
torch.cuda.empty_cache()


## 6. Start or resume training explicitly

Training samples one of the ten answers per image per epoch; frequent answers
are more likely. Image order and answer sampling are reproducible on resume.
The vision and question encoders remain frozen. The last two decoder blocks,
LM head and tied decoder embeddings train. FP16, gradient clipping, 5% warmup
and linear learning-rate decay are enabled. A short final accumulation group
is weighted correctly by its valid answer tokens.

The model does not gain OCR fusion, learned evidence selection, or a calibrated
answerability gate through this training. Those are separate project stages.


In [ ]:
RUN_TRAINING = False  # Enable only in the Colab GPU runtime.
if RUN_TRAINING:
    resume_snapshot = RESUME and (RUN_DIR / 'latest.json').exists()
    if RESUME and not resume_snapshot:
        print('No saved snapshot yet: restarting this run from the unchanged original checkpoint.')
    export = train_run(config, RUN_DIR, DATA_ROOT / 'images', scorer, resume=resume_snapshot)
    print('Export:', export)
    print(json.dumps(read_json(export.parent / 'training_summary.json'), indent=2))
else:
    print('Training disabled. Enable only after the GPU smoke check succeeds.')

## 7. Final comparison, once model selection is finished

Compare original pretrained BLIP, the existing VizWiz checkpoint, and the best
new export on identical holdout images, processing and decoding. Report the
official VQA score and answerability/answer-type breakdowns. Scores here are
not directly comparable to Vidish's 48.4% approximate score on 100 other images.
The evaluation calls BLIP directly, with no application safeguards or speech.

Use the same starting-checkpoint processor for all three models; incompatible
vocabularies are rejected. Final comparison requires completed early stopping
or maximum epochs. On a disconnect, rerun it to reuse fully saved model results.
Losses, comparison CSV, all prediction rows and PNG/PDF graphs are saved to Drive.


In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after max_epochs or early_stopping.
if RUN_FINAL_EVALUATION:
    if (RUN_DIR / 'final_comparison.json').exists():
        comparison = read_json(RUN_DIR / 'final_comparison.json')['models']
        print('Displaying the saved final comparison, without evaluating again.')
    else:
        comparison = final_compare(config, RUN_DIR, DATA_ROOT / 'images', scorer)
    print(json.dumps(comparison, indent=2))
    folder = RUN_DIR / read_json(RUN_DIR / 'final_comparison.json')['folder']
    print('Saved JSON/CSV/graphs and all question/reference/prediction rows:', folder)
    from IPython.display import display, Image
    display(Image(filename=str(folder / 'model_comparison.png')))
    with (folder / 'sample_comparison.csv').open(encoding='utf-8', newline='') as stream:
        import csv, itertools
        for sample in itertools.islice(csv.DictReader(stream), 5):
            print(json.dumps(sample, indent=2, ensure_ascii=False))
else:
    print('Final evaluation disabled. A time-budget pause must be resumed first.')

## 8. Export and use updated weights without replacing the original

The complete best-model directory is recorded in `export.json`; it contains
model.safetensors and processor/tokenizer files. Resumable delta snapshots
require the original starting checkpoint and are not standalone app models.
Keep the entire run folder for reproducibility. Never copy the new weights over
`models/blip_vizwiz`. Later import them as `models/blip_vizwiz_v2` and configure
the application to use that separate directory after reviewing the comparison.

The original checkpoint had no optimizer state. This run starts a new optimizer;
only checkpoints created by this workflow support resuming its training state.


In [ ]:
import json, zipfile
if (RUN_DIR / 'export.json').exists():
    model_dir = RUN_DIR / read_json(RUN_DIR / 'export.json')['model']
    print('Best model:', model_dir)
    MAKE_ZIP = False
    if MAKE_ZIP:
        archive = model_dir.parent / 'visionbridge_improved_model.zip'
        with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED) as z:
            for path in model_dir.iterdir():
                if path.is_file():
                    z.write(path, str(Path('blip_vizwiz_v2') / path.name))
        files.download(str(archive))
else:
    print('No trained export yet. Original checkpoint remains unchanged.')


## Optional: inspect training and development curves

Training loss is recorded per accumulated optimizer attempt. Development loss
and VQA score are recorded each epoch. Falling training loss with rising
development loss suggests overfitting. Do not use final holdout results to
choose more epochs or tune settings.

Curves are automatically saved after each development check and export, not
only displayed. Cumulative saved training-stage wall time includes validation
and checkpoint work, excluding setup, initial evaluation and final export.
Unsaved work/time after an unexpected disconnect can be lost and replayed.


In [ ]:
if (RUN_DIR / 'history.json').exists():
    from visionbridge_training import save_training_plots
    from IPython.display import display, Image
    save_training_plots(RUN_DIR, read_json(RUN_DIR / 'history.json'))
    display(Image(filename=str(RUN_DIR / 'plots/training_validation.png')))
    print('Curves are also saved on Drive:', RUN_DIR / 'plots')
    print('Loss/VQA/time rows:', RUN_DIR / 'history.csv')
else:
    print('No training history yet.')